In [ ]:
pip install sentence-transformers faiss-cpu matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.2 MB/s eta 0:00:00


In [ ]:
"""
Flow:
  1. Generates Synthetic-128D dataset (Gaussian vectors)
  2. Generates NLP-768D dataset using MS MARCO + sentence-transformers
     (falls back to synthetic 768D if MS MARCO download fails)
  3. Builds clean ground truth via brute-force FAISS Flat index
  4. Injects leakage at 5 contamination levels: 0%, 5%, 10%, 20%, 30%
  5. Evaluates Recall@1, Recall@5, Recall@10, MRR on each condition
  6. Runs random injection control experiment
  7. Runs leakage similarity threshold sensitivity experiment
  8. Runs leakage detection protocol
  9. Plots all 6 figures used in the paper
 10. Prints all tables used in the paper
"""

import numpy as np
import faiss
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import os, time, warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
RESULTS_DIR = "paper_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

FONT = 'DejaVu Serif'
plt.rcParams.update({'font.family': FONT, 'font.size': 11})

#  SECTION 1 — Dataset Generation


def make_synthetic_dataset(n_corpus=100_000, n_queries=1_000, dim=128, seed=42):
    """
    Synthetic-128D: corpus and queries drawn from unit hypersphere.
    Each vector is sampled from N(0,I) then L2-normalized.
    """
    rng = np.random.RandomState(seed)
    corpus  = rng.randn(n_corpus,  dim).astype('float32')
    queries = rng.randn(n_queries, dim).astype('float32')
    # L2-normalize
    faiss.normalize_L2(corpus)
    faiss.normalize_L2(queries)
    print(f"[Synthetic-128D] corpus={corpus.shape}, queries={queries.shape}")
    return corpus, queries


def make_nlp_dataset(n_corpus=50_000, n_queries=500, dim=768, seed=42):
    """
    NLP-768D: tries to load MS MARCO via sentence-transformers.
    Falls back to synthetic 768D Gaussian vectors if not available.
    """
    try:
        from sentence_transformers import SentenceTransformer
        from datasets import load_dataset
        print("[NLP-768D] Loading MS MARCO passages ...")
        ds = load_dataset("ms_marco", "v1.1", split="train", streaming=True)
        passages, queries_text = [], []
        for i, row in enumerate(ds):
            if i >= n_corpus + n_queries:
                break
            if i < n_corpus:
                passages.append(row['passages']['passage_text'][0][:200])
            else:
                queries_text.append(row['query'])
        model = SentenceTransformer('all-mpnet-base-v2')
        print("[NLP-768D] Encoding corpus ...")
        corpus  = model.encode(passages,      batch_size=256, show_progress_bar=True,
                               convert_to_numpy=True).astype('float32')
        print("[NLP-768D] Encoding queries ...")
        queries = model.encode(queries_text,  batch_size=256, show_progress_bar=True,
                               convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(corpus)
        faiss.normalize_L2(queries)
        print(f"[NLP-768D] corpus={corpus.shape}, queries={queries.shape}")
        return corpus, queries

    except Exception as e:
        print(f"[NLP-768D] MS MARCO unavailable ({e}). Using synthetic 768D fallback.")
        rng = np.random.RandomState(seed + 1)
        # Synthetic 768D with block structure to simulate semantic clustering
        n_clusters = 200
        centers = rng.randn(n_clusters, dim).astype('float32')
        faiss.normalize_L2(centers)
        def sample_cluster(n):
            idx = rng.randint(0, n_clusters, n)
            noise = rng.randn(n, dim).astype('float32') * 0.3
            vecs = centers[idx] + noise
            faiss.normalize_L2(vecs)
            return vecs
        corpus  = sample_cluster(n_corpus)
        queries = sample_cluster(n_queries)
        print(f"[NLP-768D fallback] corpus={corpus.shape}, queries={queries.shape}")
        return corpus, queries



#  SECTION 2 — Ground Truth Construction

def build_ground_truth(corpus, queries, K=10):
    """
    Exhaustive brute-force nearest neighbor search.
    Returns ground_truth[i] = list of K true nearest neighbor indices for query i.
    Uses inner product (= cosine similarity for normalized vectors).
    """
    dim = corpus.shape[1]
    index = faiss.IndexFlatIP(dim)   # IP = Inner Product
    index.add(corpus)
    _, I = index.search(queries, K)  # I shape: (n_queries, K)
    return I  # shape (n_queries, K)


#  SECTION 3 — Leakage Injection

def inject_leakage(corpus, queries, ground_truth,
                   contamination_rate=0.10,
                   target_sim=0.95,
                   seed=42):
    """
    Injects perturbed copies of a fraction of test queries into the corpus.

    contamination_rate : fraction of queries whose near-duplicate is injected
    target_sim         : cosine similarity between injected vector and original query
                         (controls how 'close' the leak is)

    Returns:
        contaminated_corpus : original corpus + injected vectors
        leaked_query_indices : which query indices were leaked
    """
    rng = np.random.RandomState(seed)
    n_queries, dim = queries.shape
    n_leak = int(np.ceil(contamination_rate * n_queries))

    leaked_indices = rng.choice(n_queries, size=n_leak, replace=False)

    injected_vectors = []
    for qi in leaked_indices:
        q = queries[qi].copy()
        # Add Gaussian noise orthogonal to q to achieve target cosine similarity
        # If we want sim(q, q + noise) = target_sim:
        # cos(theta) = target_sim => sin(theta) = sqrt(1 - target_sim^2)
        # scale noise to achieve this angle
        noise = rng.randn(dim).astype('float32')
        # Remove component parallel to q (make noise orthogonal)
        noise -= np.dot(noise, q) * q
        noise_norm = np.linalg.norm(noise)
        if noise_norm > 1e-8:
            noise = noise / noise_norm
        # Set magnitude: tan(theta) = sin/cos
        sin_theta = np.sqrt(max(0.0, 1.0 - target_sim**2))
        leaked_vec = target_sim * q + sin_theta * noise
        leaked_vec = leaked_vec / (np.linalg.norm(leaked_vec) + 1e-10)
        injected_vectors.append(leaked_vec)

    injected_vectors = np.array(injected_vectors, dtype='float32')
    contaminated_corpus = np.vstack([corpus, injected_vectors])
    faiss.normalize_L2(contaminated_corpus)  # re-normalize after stacking

    return contaminated_corpus, leaked_indices


def inject_random(corpus, queries, contamination_rate=0.10, seed=42):
    """
    Random injection baseline: injects random vectors unrelated to queries.
    Used to confirm that inflation is not due to corpus size increase alone.
    """
    rng = np.random.RandomState(seed + 999)
    n_queries, dim = queries.shape
    n_inject = int(np.ceil(contamination_rate * n_queries))
    random_vecs = rng.randn(n_inject, dim).astype('float32')
    faiss.normalize_L2(random_vecs)
    contaminated_corpus = np.vstack([corpus, random_vecs])
    faiss.normalize_L2(contaminated_corpus)
    return contaminated_corpus

#  SECTION 4 — FAISS Index + Retrieval

def build_ivf_index(corpus, nlist=256):
    """
    Builds FAISS IVF-Flat index with inner product similarity.
    nlist=256 Voronoi cells, standard for medium-scale benchmarks.
    """
    dim = corpus.shape[1]
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    index.train(corpus)
    index.add(corpus)
    index.nprobe = 32   # cells to visit during search (speed/recall tradeoff)
    return index


def retrieve(index, queries, K=10):
    """Returns top-K indices for each query."""
    _, I = index.search(queries, K)
    return I  # shape (n_queries, K)


#  SECTION 5 — Evaluation Metrics

def recall_at_k(retrieved, ground_truth, k):
    """
    Recall@K: fraction of ground-truth neighbors found in top-K retrieved.
    Averaged over all queries.
    """
    n_queries = retrieved.shape[0]
    gt_k = ground_truth[:, :k]  # use top-k ground truth
    scores = []
    for i in range(n_queries):
        gt_set = set(gt_k[i].tolist())
        ret_set = set(retrieved[i, :k].tolist())
        if len(gt_set) == 0:
            continue
        scores.append(len(gt_set & ret_set) / len(gt_set))
    return float(np.mean(scores)) * 100  # return as percentage


def mrr(retrieved, ground_truth):
    """
    Mean Reciprocal Rank: 1/rank of first relevant item, averaged over queries.
    """
    n_queries = retrieved.shape[0]
    rr_scores = []
    for i in range(n_queries):
        gt_set = set(ground_truth[i].tolist())
        rr = 0.0
        for rank, idx in enumerate(retrieved[i], start=1):
            if idx in gt_set:
                rr = 1.0 / rank
                break
        rr_scores.append(rr)
    return float(np.mean(rr_scores))


def evaluate(index, queries, ground_truth):
    """Evaluate all metrics in one call."""
    retrieved = retrieve(index, queries, K=10)
    return {
        'R@1':  recall_at_k(retrieved, ground_truth, 1),
        'R@5':  recall_at_k(retrieved, ground_truth, 5),
        'R@10': recall_at_k(retrieved, ground_truth, 10),
        'MRR':  mrr(retrieved, ground_truth),
    }

#  SECTION 6 — Leakage Detection Protocol


def compute_leakage_rate(corpus, queries, ground_truth, tau=0.95):
    """
    For each query, checks if any corpus vector (outside GT) has
    cosine similarity >= tau. Returns fraction of queries affected.
    """
    dim = corpus.shape[1]
    index_flat = faiss.IndexFlatIP(dim)
    index_flat.add(corpus)
    K_check = 20  # check top-20 to catch near-duplicates
    _, I = index_flat.search(queries, K_check)

    n_queries = queries.shape[0]
    leaked = 0
    for i in range(n_queries):
        gt_set = set(ground_truth[i].tolist())
        # recompute similarities for top candidates
        candidates = corpus[I[i]]
        sims = (candidates @ queries[i])  # dot product = cosine (normalized)
        for j, sim in enumerate(sims):
            if sim >= tau and I[i][j] not in gt_set:
                leaked += 1
                break
    return leaked / n_queries


#  SECTION 7 — Main Experiment Runner


CONTAMINATION_LEVELS = [0, 5, 10, 20, 30]
N_RUNS = 3  # set to 5 for paper-quality variance estimates (slower)

def run_experiment(corpus, queries, ground_truth, dataset_name, target_sim=0.95):
    """
    Runs the full contamination experiment across all levels.
    Returns dict of results.
    """
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {dataset_name}")
    print(f"{'='*60}")

    results = {level: {'R@1': [], 'R@5': [], 'R@10': [], 'MRR': []}
               for level in CONTAMINATION_LEVELS}

    for run in range(N_RUNS):
        print(f"\n  Run {run+1}/{N_RUNS}")
        for level in CONTAMINATION_LEVELS:
            rate = level / 100.0
            if rate == 0:
                working_corpus = corpus.copy()
            else:
                working_corpus, _ = inject_leakage(
                    corpus, queries, ground_truth,
                    contamination_rate=rate,
                    target_sim=target_sim,
                    seed=SEED + run * 100
                )
            index = build_ivf_index(working_corpus)
            metrics = evaluate(index, queries, ground_truth)
            for k, v in metrics.items():
                results[level][k].append(v)
            print(f"    contam={level:3d}%  R@1={metrics['R@1']:.1f}  "
                  f"R@5={metrics['R@5']:.1f}  R@10={metrics['R@10']:.1f}  "
                  f"MRR={metrics['MRR']:.3f}")

    # Compute means
    means = {}
    for level in CONTAMINATION_LEVELS:
        means[level] = {k: float(np.mean(v)) for k, v in results[level].items()}

    return means


def run_random_injection_control(corpus, queries, ground_truth, dataset_name):
    """Random injection baseline to confirm inflation is semantic, not size-based."""
    print(f"\n  Random Injection Control: {dataset_name}")
    random_results = []
    for level in CONTAMINATION_LEVELS:
        rate = level / 100.0
        if rate == 0:
            working_corpus = corpus.copy()
        else:
            working_corpus = inject_random(corpus, queries, contamination_rate=rate)
        index = build_ivf_index(working_corpus)
        metrics = evaluate(index, queries, ground_truth)
        random_results.append(metrics['R@10'])
        print(f"    contam={level:3d}%  R@10={metrics['R@10']:.1f}  (random injection)")
    return random_results


def run_similarity_threshold_experiment(corpus, queries, ground_truth, dataset_name,
                                         contamination_rate=0.20):
    """Tests how recall inflation changes with leakage similarity."""
    print(f"\n  Similarity Threshold Experiment: {dataset_name}")
    sim_thresholds = [0.92, 0.95, 0.97, 0.99]
    inflation_results = []
    # Clean baseline
    idx_clean = build_ivf_index(corpus)
    clean_r10 = evaluate(idx_clean, queries, ground_truth)['R@10']
    for tau in sim_thresholds:
        cont_corpus, _ = inject_leakage(corpus, queries, ground_truth,
                                         contamination_rate=contamination_rate,
                                         target_sim=tau, seed=SEED)
        idx = build_ivf_index(cont_corpus)
        r10 = evaluate(idx, queries, ground_truth)['R@10']
        inflation = r10 - clean_r10
        inflation_results.append(inflation)
        print(f"    sim={tau}  R@10={r10:.1f}  inflation=+{inflation:.1f} pp")
    return sim_thresholds, inflation_results, clean_r10


def run_detection_protocol(corpus, queries, ground_truth):
    """Leakage detection at multiple thresholds for clean vs. contaminated corpus."""
    print("\n  Leakage Detection Protocol ...")
    thresholds = [0.85, 0.88, 0.90, 0.92, 0.95, 0.97, 0.99]
    # Clean
    clean_rates = [compute_leakage_rate(corpus, queries, ground_truth, tau=t)
                   for t in thresholds]
    # 30% contaminated
    cont_corpus, _ = inject_leakage(corpus, queries, ground_truth,
                                     contamination_rate=0.30, target_sim=0.95)
    cont_rates = [compute_leakage_rate(cont_corpus, queries, ground_truth, tau=t)
                  for t in thresholds]
    for tau, cr, cnr in zip(thresholds, clean_rates, cont_rates):
        print(f"    tau={tau}  clean={cr:.3f}  contaminated={cnr:.3f}")
    return thresholds, clean_rates, cont_rates

#  SECTION 8 — Plotting (all 6 paper figures)

def plot_recall_both_datasets(synth_means, nlp_means):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)
    labels = ['(a) Synthetic-128D', '(b) NLP-768D']
    for ax, means, label in zip(axes, [synth_means, nlp_means], labels):
        x = CONTAMINATION_LEVELS
        r1  = [means[l]['R@1']  for l in x]
        r5  = [means[l]['R@5']  for l in x]
        r10 = [means[l]['R@10'] for l in x]
        ax.plot(x, r1,  'o-',  color='#2166ac', lw=2,   label='Recall@1',  ms=5)
        ax.plot(x, r5,  's--', color='#4dac26', lw=2,   label='Recall@5',  ms=5)
        ax.plot(x, r10, '^-',  color='#d7191c', lw=2,   label='Recall@10', ms=5)
        ax.axhline(y=r10[0], color='#d7191c', ls=':', alpha=0.5, lw=1.2)
        ax.set_xlabel('Contamination Level (%)')
        ax.set_ylabel('Recall (%)')
        ax.set_title(label, fontweight='bold')
        ax.set_xticks(x)
        ax.set_ylim(60, 102)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    plt.tight_layout()
    path = f"{RESULTS_DIR}/fig1_recall_both.png"
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Saved: {path}")


def plot_mrr(synth_means, nlp_means):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    x = CONTAMINATION_LEVELS
    s_mrr = [synth_means[l]['MRR'] for l in x]
    n_mrr = [nlp_means[l]['MRR']   for l in x]
    ax.plot(x, s_mrr, 'o-',  color='#2166ac', lw=2.5, label='Synthetic-128D', ms=6)
    ax.plot(x, n_mrr, 's--', color='#d7191c', lw=2.5, label='NLP-768D',       ms=6)
    ax.axhline(y=s_mrr[0], color='#2166ac', ls=':', alpha=0.4)
    ax.axhline(y=n_mrr[0], color='#d7191c', ls=':', alpha=0.4)
    ax.set_xlabel('Contamination Level (%)')
    ax.set_ylabel('Mean Reciprocal Rank (MRR)')
    ax.set_title('MRR vs. Contamination Level', fontweight='bold')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    path = f"{RESULTS_DIR}/fig2_mrr.png"
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Saved: {path}")


def plot_random_vs_leakage(nlp_means, nlp_random_r10):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    x = CONTAMINATION_LEVELS
    leakage_r10 = [nlp_means[l]['R@10'] for l in x]
    ax.plot(x, leakage_r10,   'o-',  color='#d7191c', lw=2.5, label='Leakage Injection (Semantic)', ms=6)
    ax.plot(x, nlp_random_r10,'s--', color='#4dac26', lw=2.5, label='Random Injection (Control)',    ms=6)
    ax.fill_between(x, nlp_random_r10, leakage_r10, alpha=0.12, color='#d7191c', label='Inflation Gap')
    ax.set_xlabel('Injection Rate (%)')
    ax.set_ylabel('Recall@10 (%)')
    ax.set_title('Semantic vs. Random Injection (NLP-768D)', fontweight='bold')
    ax.set_xticks(x)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    path = f"{RESULTS_DIR}/fig3_random_vs_leakage.png"
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Saved: {path}")


def plot_inflation_bar(sim_thresholds, inflation_values):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    colors = ['#4393c3', '#2166ac', '#f4a582', '#d6604d']
    bars = ax.bar([str(t) for t in sim_thresholds], inflation_values,
                  color=colors, edgecolor='black', lw=0.8, width=0.5)
    for bar, val in zip(bars, inflation_values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
                f'+{val:.1f} pp', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xlabel('Leakage Cosine Similarity Threshold')
    ax.set_ylabel('Recall@10 Inflation (pp)')
    ax.set_title('Recall@10 Inflation vs. Leakage Similarity\n(20% Contamination, NLP-768D)', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    path = f"{RESULTS_DIR}/fig4_inflation_bar.png"
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Saved: {path}")


def plot_metric_sensitivity(nlp_means):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    x = CONTAMINATION_LEVELS
    base_r1  = nlp_means[0]['R@1']
    base_r5  = nlp_means[0]['R@5']
    base_r10 = nlp_means[0]['R@10']
    inf_r1  = [nlp_means[l]['R@1']  - base_r1  for l in x]
    inf_r5  = [nlp_means[l]['R@5']  - base_r5  for l in x]
    inf_r10 = [nlp_means[l]['R@10'] - base_r10 for l in x]
    ax.plot(x, inf_r1,  'o-',  color='#2166ac', lw=2.5, label='Recall@1 Inflation',  ms=6)
    ax.plot(x, inf_r5,  's--', color='#4dac26', lw=2.5, label='Recall@5 Inflation',  ms=6)
    ax.plot(x, inf_r10, '^-',  color='#d7191c', lw=2.5, label='Recall@10 Inflation', ms=6)
    ax.set_xlabel('Contamination Level (%)')
    ax.set_ylabel('Inflation (percentage points)')
    ax.set_title('Metric Sensitivity to Leakage (NLP-768D)', fontweight='bold')
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    path = f"{RESULTS_DIR}/fig5_metric_sensitivity.png"
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Saved: {path}")


def plot_detection(thresholds, clean_rates, cont_rates):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    ax.plot(thresholds, [r*100 for r in clean_rates], 'o--', color='#4dac26', lw=2.5,
            label='Clean Benchmark (0%)',      ms=6)
    ax.plot(thresholds, [r*100 for r in cont_rates],  's-',  color='#d7191c', lw=2.5,
            label='30% Contaminated Benchmark', ms=6)
    ax.set_xlabel('Detection Threshold (\u03C4)')
    ax.set_ylabel('Detected Leakage Rate (%)')
    ax.set_title('Leakage Detection Protocol Performance', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    path = f"{RESULTS_DIR}/fig6_detection.png"
    plt.savefig(path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Saved: {path}")



#  SECTION 9 — Print Tables


def print_main_table(synth_means, nlp_means):
    print("\n" + "="*80)
    print("TABLE III — Retrieval Performance vs. Contamination Level")
    print("="*80)
    print(f"{'Dataset':<18} {'Contam.':<10} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'MRR':>7} {'Inflate@10':>12}")
    print("-"*80)
    for dataset_name, means in [("Synth-128D", synth_means), ("NLP-768D", nlp_means)]:
        base_r10 = means[0]['R@10']
        for level in CONTAMINATION_LEVELS:
            m = means[level]
            inflate = m['R@10'] - base_r10
            inflate_str = f"+{inflate:.1f} pp" if inflate > 0 else "—"
            label = f"{level}% (Clean)" if level == 0 else f"{level}%"
            print(f"{dataset_name:<18} {label:<10} {m['R@1']:>6.1f}% {m['R@5']:>6.1f}% "
                  f"{m['R@10']:>6.1f}% {m['MRR']:>7.3f} {inflate_str:>12}")
        print()


def print_similarity_table(sim_thresholds, inflation_values, clean_r10):
    print("="*60)
    print("TABLE IV — Recall@10 Inflation vs. Leakage Similarity")
    print("(20% Contamination, NLP-768D)")
    print("="*60)
    print(f"{'Leakage Sim.':<15} {'R@10 Contam.':>14} {'R@10 Clean':>12} {'Inflation':>12}")
    print("-"*60)
    for tau, inf in zip(sim_thresholds, inflation_values):
        cont_r10 = clean_r10 + inf
        print(f"{tau:<15.2f} {cont_r10:>13.1f}% {clean_r10:>11.1f}% {'+'+str(round(inf,1))+' pp':>12}")


#  MAIN


if __name__ == "__main__":
    t0 = time.time()

    print("\n" + "="*60)
    print("  BUILDING DATASETS")
    print("="*60)
    synth_corpus,  synth_queries  = make_synthetic_dataset()
    nlp_corpus,    nlp_queries    = make_nlp_dataset()

    print("\nBuilding ground truth (brute-force, clean data) ...")
    synth_gt = build_ground_truth(synth_corpus,  synth_queries)
    nlp_gt   = build_ground_truth(nlp_corpus,    nlp_queries)

    # ── Main contamination experiments ──
    synth_means = run_experiment(synth_corpus, synth_queries, synth_gt, "Synthetic-128D")
    nlp_means   = run_experiment(nlp_corpus,   nlp_queries,  nlp_gt,   "NLP-768D")

    # ── Random injection control ──
    print("\n" + "="*60 + "\n  RANDOM INJECTION CONTROL\n" + "="*60)
    nlp_random_r10 = run_random_injection_control(nlp_corpus, nlp_queries, nlp_gt, "NLP-768D")

    # ── Similarity threshold sensitivity ──
    print("\n" + "="*60 + "\n  SIMILARITY THRESHOLD EXPERIMENT\n" + "="*60)
    sim_thresholds, inflation_values, clean_r10 = run_similarity_threshold_experiment(
        nlp_corpus, nlp_queries, nlp_gt, "NLP-768D", contamination_rate=0.20)

    # ── Leakage detection protocol ──
    print("\n" + "="*60 + "\n  LEAKAGE DETECTION PROTOCOL\n" + "="*60)
    det_thresholds, clean_rates, cont_rates = run_detection_protocol(
        synth_corpus, synth_queries, synth_gt)

    # ── Print tables ──
    print("\n\n" + "="*80)
    print("  PAPER TABLES")
    print("="*80)
    print_main_table(synth_means, nlp_means)
    print_similarity_table(sim_thresholds, inflation_values, clean_r10)

    # ── Generate all plots ──
    print("\n\nGenerating paper figures ...")
    plot_recall_both_datasets(synth_means, nlp_means)
    plot_mrr(synth_means, nlp_means)
    plot_random_vs_leakage(nlp_means, nlp_random_r10)
    plot_inflation_bar(sim_thresholds, inflation_values)
    plot_metric_sensitivity(nlp_means)
    plot_detection(det_thresholds, clean_rates, cont_rates)

    print(f"\nAll done in {time.time()-t0:.1f}s")
    print(f"Figures saved to: ./{RESULTS_DIR}/")
    print("Files:", os.listdir(RESULTS_DIR))


  BUILDING DATASETS
[Synthetic-128D] corpus=(100000, 128), queries=(1000, 128)
[NLP-768D] Loading MS MARCO passages ...


README.md: 0.00B [00:00, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

[NLP-768D] Encoding corpus ...


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

[NLP-768D] Encoding queries ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[NLP-768D] corpus=(50000, 768), queries=(500, 768)

Building ground truth (brute-force, clean data) ...

  EXPERIMENT: Synthetic-128D

  Run 1/3
    contam=  0%  R@1=49.8  R@5=45.9  R@10=44.3  MRR=0.999
    contam=  5%  R@1=47.1  R@5=45.8  R@10=45.0  MRR=0.974
    contam= 10%  R@1=42.4  R@5=46.2  R@10=44.7  MRR=0.947
    contam= 20%  R@1=38.6  R@5=46.2  R@10=44.4  MRR=0.895
    contam= 30%  R@1=34.8  R@5=46.2  R@10=44.7  MRR=0.847

  Run 2/3
    contam=  0%  R@1=49.8  R@5=45.9  R@10=44.3  MRR=0.999
    contam=  5%  R@1=45.3  R@5=45.3  R@10=44.2  MRR=0.972
    contam= 10%  R@1=43.5  R@5=46.0  R@10=44.6  MRR=0.949
    contam= 20%  R@1=37.7  R@5=45.2  R@10=44.2  MRR=0.893
    contam= 30%  R@1=35.0  R@5=45.2  R@10=43.9  MRR=0.848

  Run 3/3
    contam=  0%  R@1=49.8  R@5=45.9  R@10=44.3  MRR=0.999
    contam=  5%  R@1=45.0  R@5=45.9  R@10=44.9  MRR=0.973
    contam= 10%  R@1=42.0  R@5=45.6  R@10=44.4  MRR=0.946
    contam= 20%  R@1=36.1  R@5=45.4  R@10=44.3  MRR=0.894
    contam= 30%  R@1=